# Chapter 6 데이터 과학 실습, 펭귄 관측 자료 다듬고 읽기

**AI교육의 이해 / 언플러그드AI교육, Chapter 6 인공지능과 데이터과학**
전인성, 광주교육대학교 컴퓨터교육과

<a href="https://colab.research.google.com/github/jinsung4069/jinsung4069.github.io/blob/main/data/lectures/ai-education-ch6-data-lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 상황
과학 동아리 학생들과 남극 펭귄의 관측 자료로 탐구 활동을 하려고 합니다. 여러 사람이 옮겨 적은 표라서 같은 종 이름이 다르게 적혀 있거나, 같은 줄이 두 번 들어가 있거나, 단위를 잘못 적은 값이 섞여 있을 수 있습니다.

## 탐구 질문
1. 몸의 측정값만으로 세 종의 펭귄을 구별할 수 있을까요?
2. 어떤 측정값이 종을 가장 잘 구별할까요?
3. 이 자료로 얻은 결론을 어디까지 일반화할 수 있을까요?

## 실습 방법
1. 위에서부터 코드 셀을 하나씩 실행합니다. 셀 왼쪽의 실행 단추를 누르거나 `Shift + Enter` 를 누릅니다.
2. 코드 안의 `____` 는 직접 채우는 빈칸입니다. 빈칸은 다섯 곳이며, 채운 뒤 실행하면 확인 메시지가 나타납니다.
3. **예상 기록** 칸은 결과를 보기 전에 먼저 적습니다. 글 셀을 두 번 누르면 편집할 수 있습니다.
4. 기록을 저장하려면 `파일 > Drive에 사본 저장` 을 먼저 선택합니다.

## 사용하는 데이터
남극 파머 기지 장기생태연구(Palmer Station LTER)에서 2007년부터 2009년까지 세 섬의 펭귄 344마리를 측정한 자료입니다(Gorman 외, 2014). R 패키지 palmerpenguins(Horst 외, 2020)가 공개한 정리본을 사용하며 이용 조건은 CC0입니다.
이 실습 파일은 **정제 연습을 위해 일부러 오류를 넣은 사본** 입니다. 종 이름 표기 차이, 몸무게 단위 오류, 중복 행이 섞여 있습니다. 원래 자료의 결측치는 그대로 남아 있습니다.

## 0단계, 준비 코드 실행
아래 셀은 실습에 필요한 도구와 확인 함수를 불러옵니다. 내용을 고치지 않고 실행만 합니다.

In [ ]:
# @title 준비 코드 (실행만 하세요) { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython

def _blank_handler(self, etype, value, tb, tb_offset=None):
    history = self.user_ns.get('In') or ['']
    if '____' in history[-1]:
        print('[빈칸 확인] 아직 채우지 않은 빈칸 ____ 이 있습니다. 빈칸을 채운 뒤 다시 실행하세요.')
        return None
    return self.showtraceback((etype, value, tb), tb_offset=tb_offset)

get_ipython().set_custom_exc((Exception,), _blank_handler)

pd.set_option('display.precision', 2)
DATA_URL = 'https://raw.githubusercontent.com/jinsung4069/jinsung4069.github.io/main/data/lectures/ai-education-ch6-penguins-practice.csv'
MEASURES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
COLORS = {'Adelie': '#e07b39', 'Chinstrap': '#8e5bb5', 'Gentoo': '#2a8c82'}
log = []

def check(n, value):
    ok = {
        1: lambda v: set(v.dropna().unique()) == {'Adelie', 'Chinstrap', 'Gentoo'},
        2: lambda v: int(v.duplicated().sum()) == 0 and len(v) < 350,
        3: lambda v: 5 < float(v) <= 2700,
        4: lambda v: v == 'flipper_length_mm',
        5: lambda v: list(v.columns) == ['Biscoe', 'Dream', 'Torgersen'],
    }
    try:
        passed = ok[n](value)
    except Exception:
        passed = False
    print(f'[빈칸 {n}] ' + ('확인했습니다. 다음 단계로 넘어가세요.' if passed else '결과가 예상과 다릅니다. 안내 문장을 다시 읽고 빈칸을 고쳐 보세요.'))

## 1단계, 문제 인식
데이터를 모으기 전에 무엇을 알고 싶은지, 그 질문에 답하려면 어떤 데이터가 필요한지 정합니다.

### 예상 기록 (데이터를 보기 전에 적기)
| 항목 | 기록 |
|---|---|
| 해결하려는 질문 | |
| 필요한 데이터 (무엇을, 어떤 단위로) | |
| 종을 가장 잘 구별할 것 같은 측정값과 이유 | |

## 2단계, 데이터 수집
공개된 CSV 파일을 불러옵니다. 행은 펭귄 한 마리, 열은 측정 항목입니다. 처음 몇 줄과 각 열의 자료형을 확인합니다.

In [ ]:
df = pd.read_csv(DATA_URL)
print(f'행 {len(df)}개, 열 {df.shape[1]}개')
display(df.head(8))
df.info()

| 열 이름 | 뜻 |
|---|---|
| species | 종 (Adelie, Chinstrap, Gentoo) |
| island | 관측한 섬 (Biscoe, Dream, Torgersen) |
| bill_length_mm, bill_depth_mm | 부리의 길이와 두께 (mm) |
| flipper_length_mm | 날개 길이 (mm) |
| body_mass_g | 몸무게 (g) |
| sex, year | 성별, 관측 연도 |

## 3단계, 데이터 다듬기
분석에 앞서 표기 차이, 중복, 단위 오류, 결측치를 차례로 찾아 고칩니다. 고친 내용은 기록으로 남겨야 다른 사람이 같은 결과를 다시 만들 수 있습니다.

### 3-1. 같은 값의 다른 표기

In [ ]:
print(df['species'].value_counts())

In [ ]:
# 빈칸 1, 앞뒤 공백을 지운 뒤 첫 글자만 대문자로 바꾸는 문자열 함수 이름을 적습니다. (힌트: capital...)
before = df['species'].nunique()
df['species'] = df['species'].str.strip().str.____()

log.append(f'종 이름 표기 통일, 서로 다른 값 {before}개를 {df["species"].nunique()}개로 정리')
print(df['species'].value_counts())
check(1, df['species'])

### 3-2. 중복 행
같은 펭귄의 기록이 두 번 들어가면 그 개체가 평균과 그래프에 두 배로 반영됩니다.

In [ ]:
print('완전히 같은 행의 수', df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values(MEASURES).head(6))

In [ ]:
# 빈칸 2, 중복 행을 지우는 함수 이름을 적습니다. (힌트: drop_...)
before = len(df)
df = df.____().reset_index(drop=True)

log.append(f'중복 행 삭제, {before}행에서 {len(df)}행으로')
print(f'남은 행 {len(df)}개')
check(2, df)

### 3-3. 단위 오류와 이상한 값
요약 통계에서 최솟값과 최댓값을 먼저 봅니다. 성체 펭귄의 몸무게가 몇 g인데 기록된 값이 이상하게 작은 행이 있나요?

In [ ]:
display(df[MEASURES].describe())
print('몸무게가 작은 순서 다섯 행')
display(df.nsmallest(5, 'body_mass_g'))

In [ ]:
# 빈칸 3, kg으로 잘못 적었다고 판단할 몸무게 기준값(g)을 적습니다. 이 값보다 작으면 1000을 곱합니다.
threshold = ____

wrong = df['body_mass_g'] < threshold
print(f'기준 {threshold} g 미만으로 찾은 행 {wrong.sum()}개')
df.loc[wrong, 'body_mass_g'] = df.loc[wrong, 'body_mass_g'] * 1000
log.append(f'몸무게 단위 오류 {wrong.sum()}행을 g 단위로 수정')
check(3, threshold)

### 3-4. 결측치
측정값이 비어 있는 행을 찾습니다. 분석의 목적에 따라 지울지, 채울지, 그대로 둘지 결정합니다.

In [ ]:
print(df.isna().sum())
display(df[df[MEASURES].isna().any(axis=1)])

In [ ]:
before = len(df)
df = df.dropna(subset=MEASURES).reset_index(drop=True)
log.append(f'네 측정값이 모두 비어 있는 행 삭제, {before}행에서 {len(df)}행으로 (성별 결측은 유지)')

print('정제 기록')
for i, line in enumerate(log, 1):
    print(f'{i}. {line}')
print(f'\n분석에 쓸 자료 {len(df)}행, 성별이 비어 있는 행 {df["sex"].isna().sum()}개')

### 해석 기록
- 성별이 비어 있는 행은 왜 지우지 않았을까요? 성별을 비교하는 분석이라면 어떻게 처리해야 할까요?
- 3-3에서 기준값을 정할 때 무엇을 근거로 삼았나요? 기준을 잘못 정하면 어떤 일이 생길까요?

## 4단계, 데이터 시각화
### 예상 기록 (그래프를 보기 전에 적기)
- 날개 길이로 세 종을 나누면 어느 종이 다른 두 종과 가장 뚜렷하게 구별될까요?

In [ ]:
# 빈칸 4, 종별 분포를 그릴 열 이름을 적습니다. 날개 길이 열입니다.
column = '____'

check(4, column)
plt.figure(figsize=(7, 3.6))
for name, part in df.groupby('species'):
    plt.hist(part[column], bins=18, alpha=0.6, label=name, color=COLORS[name])
plt.xlabel(column); plt.ylabel('count'); plt.legend(); plt.title('Distribution by species')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4.5))
for name, part in df.groupby('species'):
    plt.scatter(part['bill_length_mm'], part['bill_depth_mm'], s=18, alpha=0.75, label=name, color=COLORS[name])
plt.xlabel('bill_length_mm'); plt.ylabel('bill_depth_mm'); plt.legend(); plt.title('Bill length and depth')
plt.show()

## 5단계, 데이터 분석
### 5-1. 종별 평균

In [ ]:
display(df.groupby('species')[MEASURES].mean().round(1))
print(df['species'].value_counts())

### 5-2. 전체로 본 관계와 종별로 본 관계
부리 길이와 부리 두께의 상관계수를 전체 자료와 종별 자료에서 각각 계산합니다. 두 결과의 방향이 같은가요?

In [ ]:
overall = df['bill_length_mm'].corr(df['bill_depth_mm'])
print(f'전체 자료의 상관계수 {overall:.2f}')
for name, part in df.groupby('species'):
    print(f'{name:10s} 상관계수 {part["bill_length_mm"].corr(part["bill_depth_mm"]):.2f}')

### 5-3. 자료는 어디에서 왔을까

In [ ]:
# 빈칸 5, 종과 함께 표로 비교할 열 이름을 적습니다. 관측한 섬이 적힌 열입니다.
table = pd.crosstab(df['species'], df['____'])

display(table)
check(5, table)

### 해석 기록
- 5-2에서 전체 자료와 종별 자료의 상관 방향이 다른 이유는 무엇일까요?
- 5-3의 표를 보면 Chinstrap과 Gentoo는 각각 어느 섬에서만 관측되었나요? 이 자료로 "섬을 알면 종을 알 수 있다"고 결론 내려도 될까요?
- 이 자료에 없는 섬이나 다른 해의 펭귄에게도 같은 결론을 적용할 수 있을까요?

## 6단계, 문제 해결과 품질 점검
1단계의 질문에 답하고, 데이터 품질을 점검한 결과를 기록합니다. 확인하지 못한 사항은 추측으로 채우지 않고 **미확인** 으로 남깁니다.

| 항목 | 기록 |
|---|---|
| 질문에 대한 답과 근거가 된 그래프나 표 | |
| 정확성 (단위 오류, 잘못 적은 값) | |
| 완전성 (결측치와 처리 방법) | |
| 일관성 (표기 차이, 중복) | |
| 대표성 (섬, 연도, 종별 표본 수) | |
| 결론을 적용할 수 있는 범위 | |

## 선택 확장
연도와 섬에 따라 관측한 펭귄 수가 어떻게 다른지 확인하고, 성별이 비어 있는 행이 어느 종에 몰려 있는지 살펴봅니다.

In [ ]:
display(pd.crosstab([df['island'], df['species']], df['year'], margins=True))
print('성별이 비어 있는 행의 종')
print(df[df['sex'].isna()]['species'].value_counts())

---
자료 출처: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus Pygoscelis). PLoS ONE, 9(3), e90081. Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). palmerpenguins, CC0. 실습 파일의 표기 차이, 단위 오류, 중복 행은 정제 연습을 위해 넣은 것이며 원자료에는 없습니다.